# Fine-tune PP-OCRv6 recognition on handwritten C code (Colab)

Clean, re-runnable version of the working sequence. Trains `PP-OCRv6_medium_rec`
on the line-crop dataset built by `evaluators.build_recognition_dataset`.

**Result this produced (2026-08-30):** held-out CER `clean_ws` **0.274 -> 0.126**
(stock vs fine-tuned, same 20-image test set). See `docs/ocr/EVALUATION.md`.

**Note on the repo path:** the original session got the PaddleOCR repo via
`paddlex --install`, whose dependency install is broken on Colab's Python 3.13.
This clean notebook uses `git clone` instead (same repo, `/content/PaddleOCR`) and
runs `tools/train.py` directly. Full rationale: `docs/ocr/COLAB_SETUP_WORKING.md`.

**Before running:** Runtime -> Change runtime type -> **GPU** (T4 is enough).

## Method: transfer learning from PP-OCRv6

We do **not** train a recognizer from scratch. We start from PaddleOCR's
`PP-OCRv6_medium_rec` weights -- pretrained on large-scale **printed / scene
text** -- and fine-tune them on our handwritten C-code line crops. The base
model already knows the general "read text" task (strokes, character shapes,
left-to-right decoding); fine-tuning only teaches it our writers' handwriting
and C vocabulary. This is why a small dataset (~2.7k crops) is enough: we
specialize an already-competent reader instead of building one from zero.

**Result this produced (2026-08-30):** held-out CER **0.274 -> 0.126** (-54%),
measured separately by `evaluators/evaluate_cer` on the untouched `samples/`
test set -- see the "How we know it improved" note near the end.


## Part A - Environment

In [ ]:
!nvidia-smi

In [ ]:
# PaddlePaddle GPU build is not on plain PyPI - use Paddle's own index (cu130 = CUDA 13.0).
!python -m pip install -q paddlepaddle-gpu==3.3.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu130/

In [ ]:
import paddle
print(paddle.__version__)
paddle.utils.run_check()

In [ ]:
# Clean way to get the repo that contains tools/train.py (bypasses the broken paddlex installer).
!git clone --depth 1 https://github.com/PaddlePaddle/PaddleOCR /content/PaddleOCR

In [ ]:
# The two extra runtime deps tools/train.py needs beyond Colab's base image.
# If a later run reports ModuleNotFoundError: X, just `!pip install -q X` and re-run.
!pip install -q lmdb rapidfuzz pyclipper

## Part B - Upload the dataset

Upload a `recognition_dataset*.zip` built locally by
`evaluators.build_recognition_dataset` (or `build_crosswriter_dataset`).

**To reproduce the RELEASED model (v1):** upload
`recognition_dataset_v1_release.zip` — the exact 2,491-train / 278-val crop
set that produced the shipped model (CER 0.274 -> 0.126). Expect ~2491/278
below, and a final CER near 0.126.

The two cells below auto-detect whatever `recognition_dataset*.zip` you
upload — no need to rename it, and re-uploads with `(1)`/`(2)` suffixes are
handled.


In [ ]:
from google.colab import files
import glob, os
for f in glob.glob('recognition_dataset*.zip'):   # clear stale Colab copies so the name stays clean
    os.remove(f)
uploaded = files.upload()
ZIP = next(iter(uploaded))     # the exact filename Colab saved, whatever it is
print('uploaded as:', ZIP)


In [ ]:
import shutil
shutil.rmtree('/content/datasets', ignore_errors=True)   # clear any stale extraction
!unzip -q "{ZIP}" -d /content/
!wc -l /content/datasets/recognition/train.txt /content/datasets/recognition/val.txt


### Dataset sanity + stats (for the record)

Confirms dataset shape and, critically, that **no page appears in both the
train and val splits** (page-based split -> no leakage). The length
distribution justifies the `max_text_length` 25 -> 100 patch below with data,
not assertion.


In [ ]:
import re, statistics
from pathlib import Path
DATA = "/content/datasets/recognition"
def read_labels(p): return [l.split("\t",1) for l in Path(p).read_text().splitlines() if "\t" in l]
def page_of(i): return re.sub(r"_line\d+$", "", Path(i).stem)
train, val = read_labels(f"{DATA}/train.txt"), read_labels(f"{DATA}/val.txt")
tp, vp = {page_of(i) for i,_ in train}, {page_of(i) for i,_ in val}
lengths = [len(t) for _,t in train+val]; over25 = sum(L>25 for L in lengths)
print(f"line crops:   train={len(train)}  val={len(val)}  total={len(train)+len(val)}")
print(f"source pages: train={len(tp)}  val={len(vp)}  total={len(tp|vp)}")
print(f"page leakage: {len(tp&vp)} pages in BOTH splits  <-- MUST be 0")
print(f"label length: min={min(lengths)} max={max(lengths)} mean={statistics.mean(lengths):.1f}")
print(f"              {over25} lines ({100*over25/len(lengths):.1f}%) > 25 chars -> why max_text_length 25->100")


In [ ]:
import matplotlib.pyplot as plt

lengths = [len(t) for _, t in train + val]
plt.figure(figsize=(8, 4))
plt.hist(lengths, bins=40, color="#4C72B0", edgecolor="white")
plt.axvline(25,  color="crimson", linestyle="--", lw=2, label="old max_text_length = 25 (drops 30.3%)")
plt.axvline(100, color="green",   linestyle="--", lw=2, label="patched max_text_length = 100")
plt.xlabel("label length (characters)")
plt.ylabel("number of line crops")
plt.title(f"Line-crop label length distribution (n={len(lengths)})")
plt.legend(); plt.tight_layout()
plt.savefig("/content/label_length_hist.png", dpi=150)   # so you can download it for slides
plt.show()


## Part C - Pretrained weights + config fix (Cell A)

Download the pretrained weights locally (native PaddleOCR won't fetch a URL for
`Global.pretrained_model`) and raise `max_text_length` 25 -> 100 (at 25, ~30% of
our lines, up to 92 chars, would be silently dropped).

In [ ]:
import os, subprocess

REPO = "/content/PaddleOCR"
CFG = f"{REPO}/configs/rec/PP-OCRv6/PP-OCRv6_medium_rec.yml"

# 1) pretrained weights (~232 MB; a tiny size here means the URL failed)
url = "https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/PP-OCRv6_medium_rec_pretrained.pdparams"
dst = "/content/PP-OCRv6_medium_rec_pretrained.pdparams"
subprocess.run(["wget", "-q", "-O", dst, url], check=True)
print("pretrained bytes:", os.path.getsize(dst))

# 2) max_text_length 25 -> 100 at the YAML anchor (propagates everywhere)
with open(CFG) as f:
    txt = f.read()
assert "&max_text_length 25" in txt, "anchor not found - config changed?"
with open(CFG, "w") as f:
    f.write(txt.replace("&max_text_length 25", "&max_text_length 100"))
print("max_text_length patched to 100")

In [ ]:
# Transparency + reproducibility: show the FULL training config actually used
# (optimizer, LR schedule, architecture) and pin the exact code versions.
!cat {CFG}
!git -C /content/PaddleOCR rev-parse HEAD          # exact PaddleOCR commit
import paddle; print("paddlepaddle:", paddle.__version__)


## Part D - Train (Cell B)

Overrides: `epoch_num=40`, single-GPU (`distributed=false`), transfer-learn from
the pretrained weights, `ratio_list=[1.0]` (config default 0.5 would use half the
crops), `num_workers=2` (Colab has ~2 CPUs). Training has started when the
tracebacks stop and `ppocr INFO: epoch: ... loss:` lines appear.

### Hyperparameters and why

Base recipe comes from PaddleOCR's own `PP-OCRv6_medium_rec.yml` (Adam, Cosine
LR peak 5e-4 with 5-epoch warmup, L2 3e-5, SVTR_LCNet architecture, batch
size 64) -- unchanged. We override only:

| Override | Why |
|---|---|
| `epoch_num=40` | enough passes for our data size |
| `distributed=false` | single T4 GPU, not multi-card |
| `Train.dataset.ratio_list=[1.0]` | config default 0.5 would use only half our crops |
| `Train/Eval.loader.num_workers=2` | Colab has ~2 CPUs; default 8 stalls |
| `max_text_length: 25 -> 100` (Part C) | ~30% of our lines exceed 25 chars and would be silently dropped |

Full pinned copy of this config: `ocr_feature/notebooks/PP-OCRv6_medium_rec.yml`.


In [ ]:
%cd /content/PaddleOCR
!python tools/train.py -c configs/rec/PP-OCRv6/PP-OCRv6_medium_rec.yml \
  -o Global.epoch_num=40 Global.distributed=false \
     Global.pretrained_model=/content/PP-OCRv6_medium_rec_pretrained \
     Global.save_model_dir=/content/output/PP-OCRv6_medium_rec \
     Global.save_epoch_step=5 Global.eval_batch_step=[0,200] \
     Train.dataset.data_dir=/content/datasets/recognition \
     Train.dataset.label_file_list=[/content/datasets/recognition/train.txt] \
     Train.dataset.ratio_list=[1.0] \
     Train.loader.num_workers=2 \
     Eval.dataset.data_dir=/content/datasets/recognition \
     Eval.dataset.label_file_list=[/content/datasets/recognition/val.txt] \
     Eval.loader.num_workers=2 2>&1 | tee /content/train_log.txt


### Training curve (convergence + no-overfitting evidence)

Reads `/content/train_log.txt` (created by the `tee` on the training command
above) and plots loss + accuracy over training. The key healthy sign for a
panel: **validation accuracy keeps rising to the final epochs** rather than
peaking early and diverging. Note the accuracy here is strict **whole-line
exact match** (much harsher than CER) -- do not confuse it with the 0.126 CER
result.


In [ ]:
import re
from pathlib import Path
import matplotlib.pyplot as plt

LOG = "/content/train_log.txt"   # produced by: ...train.py ... 2>&1 | tee /content/train_log.txt

steps, loss, tr_acc = [], [], []
ev_steps, ev_acc = [], []
last_step = 0
for line in Path(LOG).read_text().splitlines():
    if "global_step:" in line:                                   # training line
        s = re.search(r"global_step: (\d+)", line)
        l = re.search(r"(?<![A-Za-z])loss: ([\d.]+)", line)       # total loss (not CTC/NRTR)
        a = re.search(r"(?<![a-z_])acc: ([\d.]+)", line)          # train-batch acc
        if s:
            last_step = int(s.group(1))
            if l and a:
                steps.append(last_step); loss.append(float(l.group(1))); tr_acc.append(float(a.group(1)))
    elif "cur metric" in line:                                    # validation-eval line
        a = re.search(r"acc: ([\d.]+)", line)
        if a:
            ev_steps.append(last_step); ev_acc.append(float(a.group(1)))

print(f"parsed {len(steps)} training points, {len(ev_acc)} eval points")

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 7), sharex=True)
ax1.plot(steps, loss, color="#C44E52", lw=1.2)
ax1.set_ylabel("training loss (CTC + NRTR)")
ax1.set_title("Fine-tuning training curve — PP-OCRv6_medium_rec on handwritten C code",
              fontsize=12, fontweight="bold")
ax1.spines[["top", "right"]].set_visible(False)
ax2.plot(steps, tr_acc, color="#8C8C8C", lw=1.0, alpha=0.8, label="train-batch accuracy")
ax2.plot(ev_steps, ev_acc, color="#2E7D32", lw=2, marker="o", ms=4,
         label="validation accuracy (exact-line match, held-out val.txt)")
ax2.set_xlabel("global step (40 epochs)"); ax2.set_ylabel("sequence accuracy")
ax2.legend(loc="lower right"); ax2.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("/content/training_curve.png", dpi=160)
plt.show()


## Part E - Export + download

Export the best checkpoint to inference format, then download it. Unzip locally
into `ocr_feature/models/fine_tuned_rec/inference/` and point
`core/ocr_pipeline.py` at it (see `docs/ocr/COLAB_SETUP_WORKING.md` Part E).

In [ ]:
%cd /content/PaddleOCR
!python tools/export_model.py -c configs/rec/PP-OCRv6/PP-OCRv6_medium_rec.yml \
  -o Global.pretrained_model=/content/output/PP-OCRv6_medium_rec/best_accuracy \
     Global.save_inference_dir=/content/inference/PP-OCRv6_medium_rec

In [ ]:
import shutil
from google.colab import files
shutil.make_archive("/content/fine_tuned_rec_model", "zip", "/content/inference/PP-OCRv6_medium_rec")
files.download("/content/fine_tuned_rec_model.zip")

## How we know it improved

The `acc` / `norm_edit_dis` printed during training are the internal signal on
`val.txt`. The **headline result -- held-out CER 0.274 -> 0.126 (-54%)** -- is
measured **separately** by `evaluators/evaluate_cer` against the untouched
`samples/` test set (20 images the model never trained on), comparing the stock
model vs. this fine-tuned one. That test set is the honest, leakage-free
evidence -- not the in-notebook val metric. The before/after CER graph lives
with that local evaluation, not here, because Colab never computed it.


## Defense-prep Q&A

Moved to `docs/ocr/DEFENSE_PREP.md` § 11 (writer-disjointness, the
cross-writer experiment, WER, hyperparameter choices) -- keeps this notebook
focused on the training pipeline itself.
